# Hanomi Feature Recognition — Kaggle Training

End-to-end training, evaluation, ablations, and baselines for the B-Rep subgraph model.
Designed for Kaggle P100/T4 GPUs. All output paths are tracked via variables — re-running
cells is safe because `RUN_DIR` auto-increments to the next available slot.

### Step 1: Environment Setup

In [2]:
import os
import sys
import glob
import shutil
from pathlib import Path

REPO_MOUNT_PATH = Path("/kaggle/input/datasets/anmolsen24/hanomi-repo")
WORKING_REPO_PATH = Path("/kaggle/working/hanomi-repo")


def setup_workspace():
    current_root = Path.cwd()
    if (current_root / "scripts" / "train.py").exists():
        print(f"[✓] Repository present at: {current_root}")
        return True
    if (WORKING_REPO_PATH / "scripts" / "train.py").exists():
        os.chdir(WORKING_REPO_PATH)
        print(f"[✓] Working dir: {os.getcwd()}")
        return True
    if REPO_MOUNT_PATH.exists():
        if not WORKING_REPO_PATH.exists():
            shutil.copytree(REPO_MOUNT_PATH, WORKING_REPO_PATH)
            print(f"[✓] Copied repo to {WORKING_REPO_PATH}")
        os.chdir(WORKING_REPO_PATH)
        print(f"[✓] Working dir: {os.getcwd()}")
        return True
    print(f"[!] Repo not found locally or at {REPO_MOUNT_PATH}. Attach the dataset or upload repo contents.")
    return False


if setup_workspace():
    sys.path.insert(0, str(Path.cwd()))

# Auto-discover H5 dataset directory.
# Accepts both naming conventions: *_MFCAD++.h5 (local) and *_MFCAD.h5 (Kaggle upload).
NAMING_VARIANTS = [
    {"training_MFCAD++.h5", "val_MFCAD++.h5", "test_MFCAD++.h5"},
    {"training_MFCAD.h5",   "val_MFCAD.h5",   "test_MFCAD.h5"},
]
H5_DATASET_PATH = None
TEST_SET_FILE = None
for candidate in sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/*.h5", recursive=True)}):
    names = {p.name for p in candidate.glob("*.h5")}
    for variant in NAMING_VARIANTS:
        if variant.issubset(names):
            H5_DATASET_PATH = str(candidate)
            TEST_SET_FILE = str(candidate / next(f for f in variant if f.startswith("test")))
            break
    if H5_DATASET_PATH:
        break

if H5_DATASET_PATH:
    print(f"[✓] H5 dataset : {H5_DATASET_PATH}")
    print(f"    test file  : {TEST_SET_FILE}")
else:
    print("[!] Could not locate H5 directory with training/val/test files.")

[✓] Working dir: /kaggle/working/hanomi-repo
[✓] H5 dataset : /kaggle/input/datasets/anmolsen24/mfcad-plus-plus
    test file  : /kaggle/input/datasets/anmolsen24/mfcad-plus-plus/test_MFCAD.h5


### Step 2: Install Dependencies

Kaggle ships PyTorch pre-installed. PyG sparse dependencies must be installed from the
version-matched wheel index (`torch-X.Y.Z+cuABC`) — a bare `pip install torch-geometric`
will silently omit `torch_scatter` / `torch_sparse` and cause import errors at runtime.

In [8]:
import subprocess
import sys

try:
    import torch
    import torch_geometric
    import yaml
    print(f"[✓] torch {torch.__version__}, torch_geometric {torch_geometric.__version__} already installed.")
except ImportError:
    import torch
    torch_ver = torch.__version__.split("+")[0]          # e.g. "2.1.2"
    cuda_tag  = ("cu" + torch.version.cuda.replace(".", "")) if torch.cuda.is_available() else "cpu"
    pyg_url   = f"https://data.pyg.org/whl/torch-{torch_ver}+{cuda_tag}.html"
    print(f"[*] Installing torch-geometric for torch={torch_ver}, {cuda_tag} ...")
    print(f"    Wheel index: {pyg_url}")
    # Install core PyG first, then sparse deps from the version-matched index.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch-geometric"])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "torch_scatter", "torch_sparse", "torch_cluster", "torch_spline_conv",
        "-f", pyg_url,
    ])
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "PyYAML", "wandb", "networkx", "scikit-learn", "h5py", "tqdm", "pandas", "psutil",
    ])
    print("[✓] Installation complete.")

[*] Installing torch-geometric for torch=2.10.0, cu128 ...
    Wheel index: https://data.pyg.org/whl/torch-2.10.0+cu128.html
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 35.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 98.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 112.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 95.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 51.6 MB/s eta 0:00:00
[✓] Installation complete.


In [4]:
# GPU availability check — must pass before proceeding.
import torch

if torch.cuda.is_available():
    prop = torch.cuda.get_device_properties(0)
    print(f"[✓] GPU  : {prop.name}")
    print(f"    VRAM : {prop.total_memory / 1e9:.1f} GB")
    print(f"    CUDA : {torch.version.cuda}")
else:
    print("[!] No GPU — training on CPU will be extremely slow.")
    print("    Enable GPU: Runtime → Change runtime type → GPU")

[✓] GPU  : Tesla T4
    VRAM : 15.6 GB
    CUDA : 12.8


### Step 3: Phase 1 Training — Counterbored Holes

Pre-compute the output directory so all subsequent cells can reference `RUN_DIR` and
`CHECKPOINT` without re-scanning the filesystem.

In [4]:
from pathlib import Path

def _next_run_dir(base="results/runs"):
    """Return the next non-existent results/runs/run_NNN path."""
    base_path = Path(base)
    for n in range(1, 1000):
        c = base_path / f"run_{n:03d}"
        if not c.exists():
            return str(c)
    return str(base_path / "run_999")


RUN_DIR    = _next_run_dir()
CHECKPOINT = f"{RUN_DIR}/checkpoints/best.pt"
EVAL_DIR   = f"{RUN_DIR}/eval"
print(f"[*] Phase 1 output : {RUN_DIR}")

[*] Phase 1 output : results/runs/run_001


In [5]:
if not H5_DATASET_PATH:
    print("[!] Halting: No H5 dataset found.")
else:
    !python scripts/train.py \
        --config configs/counterbored_hole.yaml \
        --h5_dir {H5_DATASET_PATH} \
        --output_dir {RUN_DIR} \
        --seed 42 \
        --early_stopping

23:25:33 | INFO | train | Output dir: results/runs/run_001
23:25:33 | INFO | train | Device: cuda
23:25:33 | INFO | train | Feature types: ['through_hole', 'blind_hole'] → label IDs: [1, 12]
23:37:28 | INFO | train | Dataset sizes — train: 19290  val: 4118  test: 4108
23:37:28 | INFO | train | Model params: 116,634
23:37:28 | INFO | train | Early stopping enabled: patience=10, min_delta=0.0010
23:38:33 | INFO | train | Epoch   1/50 | train_loss=2.7060 | val_loss=1.2252 | val_acc=0.627 | lr=2.00e-04 | 65.3s
23:38:33 | INFO | train |   ✓ Saved best checkpoint (val_loss=1.2252)
23:39:38 | INFO | train | Epoch   2/50 | train_loss=0.9876 | val_loss=0.8826 | val_acc=0.704 | lr=4.00e-04 | 64.5s
23:39:38 | INFO | train |   ✓ Saved best checkpoint (val_loss=0.8826)
23:40:42 | INFO | train | Epoch   3/50 | train_loss=0.7054 | val_loss=0.6597 | val_acc=0.768 | lr=6.00e-04 | 64.3s
23:40:42 | INFO | train |   ✓ Saved best checkpoint (val_loss=0.6597)
23:41:47 | INFO | train | Epoch   4/50 | train_l

### Step 4: Phase 1 Evaluation

In [6]:
if not os.path.exists(CHECKPOINT):
    print(f"[!] No checkpoint at {CHECKPOINT} — run training first.")
elif not TEST_SET_FILE:
    print("[!] No test H5 file — check dataset mount.")
else:
    !python scripts/evaluate.py \
        --weights {CHECKPOINT} \
        --h5_file {TEST_SET_FILE} \
        --config configs/counterbored_hole.yaml \
        --results_dir {EVAL_DIR}

Using device: cuda
/kaggle/working/hanomi-repo/scripts/evaluate.py:128: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  full_loader = DataLoader(full_dataset, batch_size=args.batch_size, shuffle=False)
Loaded 8632 graphs for evaluation.

--- Face-Level Segmentation Report ---
Segmentation Eval: 100%|███████████████████| 8632/8632 [00:24<00:00, 357.25it/s]

--- Face-Level Segmentation Report ---
              precision    recall  f1-score   support

           0       0.80      0.68      0.73      3487
           1       0.98      0.99      0.99      3723
           2       0.88      0.77      0.83     10965
           3       0.84      0.85      0.84     14723
           4       0.88      0.94      0.91     21871
           5       0.85      0.86      0.86      2450
           6       0.87      0.89      0.88      3935
           7       0.95      0.96      0.95      1327
           8       0.69      0.56      0.62      6797
           9       0.95      

### Step 5: Phase 2 — Extensibility Fine-Tuning

Fine-tune from the Phase 1 checkpoint to add `rectangular_pocket`. Demonstrates that
new feature types are learned without retraining from scratch.

In [7]:
PHASE2_DIR        = _next_run_dir()
PHASE2_CHECKPOINT = f"{PHASE2_DIR}/checkpoints/best.pt"
PHASE2_EVAL_DIR   = f"{PHASE2_DIR}/eval"
print(f"[*] Phase 2 output : {PHASE2_DIR}")

if not os.path.exists(CHECKPOINT):
    print(f"[!] Phase 1 checkpoint not found at {CHECKPOINT}")
elif not H5_DATASET_PATH:
    print("[!] No H5 dataset found for Phase 2.")
else:
    !python scripts/train.py \
        --config configs/extensibility_v2.yaml \
        --h5_dir {H5_DATASET_PATH} \
        --checkpoint {CHECKPOINT} \
        --output_dir {PHASE2_DIR} \
        --early_stopping

[*] Phase 2 output : results/runs/run_002
00:49:46 | INFO | train | Output dir: results/runs/run_002
00:49:46 | INFO | train | Device: cuda
00:49:46 | INFO | train | Feature types: ['through_hole', 'blind_hole', 'rectangular_pocket'] → label IDs: [1, 12, 14]
01:02:09 | INFO | train | Dataset sizes — train: 25451  val: 5430  test: 5438
01:05:03 | INFO | train | Epoch   2/30 | train_loss=0.1920 | val_loss=0.1845 | val_acc=0.933 | lr=3.00e-04 | 86.3s
01:05:03 | INFO | train |   ✓ Saved best checkpoint (val_loss=0.1845)
01:06:30 | INFO | train | Epoch   3/30 | train_loss=0.1940 | val_loss=0.1832 | val_acc=0.933 | lr=2.99e-04 | 86.3s
01:06:30 | INFO | train |   ✓ Saved best checkpoint (val_loss=0.1832)
01:07:56 | INFO | train | Epoch   4/30 | train_loss=0.1887 | val_loss=0.1886 | val_acc=0.932 | lr=2.96e-04 | 86.7s
01:09:23 | INFO | train | Epoch   5/30 | train_loss=0.1887 | val_loss=0.1803 | val_acc=0.935 | lr=2.92e-04 | 86.3s
INFO:train:Epoch   5/30 | train_loss=0.1887 | val_loss=0.1803 |

### Step 6: Phase 2 Evaluation

In [18]:
from pathlib import Path
import glob, os

def _last_run_dir(base="results/runs"):
  runs = sorted(Path(base).glob("run_*"))
  return str(runs[-1]) if runs else None

def _second_last_run_dir(base="results/runs"):
  runs = sorted(Path(base).glob("run_*"))
  return str(runs[-2]) if len(runs) >= 2 else None

RUN_DIR    = _last_run_dir() if _second_last_run_dir() is None else _second_last_run_dir()
PHASE2_DIR = _last_run_dir()

CHECKPOINT        = f"{RUN_DIR}/checkpoints/best.pt"
EVAL_DIR          = f"{RUN_DIR}/eval"
PHASE2_CHECKPOINT = f"{PHASE2_DIR}/checkpoints/best.pt"
PHASE2_EVAL_DIR   = f"{PHASE2_DIR}/eval"

# Rediscover H5 dataset
NAMING_VARIANTS = [
  {"training_MFCAD++.h5", "val_MFCAD++.h5", "test_MFCAD++.h5"},
  {"training_MFCAD.h5",   "val_MFCAD.h5",   "test_MFCAD.h5"},
]
H5_DATASET_PATH = None
TEST_SET_FILE   = None
for candidate in sorted({Path(p).parent for p in glob.glob("/kaggle/input/**/*.h5", recursive=True)}):
  names = {p.name for p in candidate.glob("*.h5")}
  for variant in NAMING_VARIANTS:
      if variant.issubset(names):
          H5_DATASET_PATH = str(candidate)
          TEST_SET_FILE   = str(candidate / next(f for f in variant if f.startswith("test")))
          break
  if H5_DATASET_PATH:
      break

print(f"RUN_DIR    = {RUN_DIR}")
print(f"PHASE2_DIR = {PHASE2_DIR}")
print(f"CHECKPOINT exists        : {os.path.exists(CHECKPOINT)}")
print(f"PHASE2_CHECKPOINT exists : {os.path.exists(PHASE2_CHECKPOINT)}")
print(f"H5_DATASET_PATH = {H5_DATASET_PATH}")


RUN_DIR    = results/runs/run_001
PHASE2_DIR = results/runs/run_002
CHECKPOINT exists        : True
PHASE2_CHECKPOINT exists : True
H5_DATASET_PATH = /kaggle/input/datasets/anmolsen24/mfcad-plus-plus


In [ ]:
if not os.path.exists(PHASE2_CHECKPOINT):
    print(f"[!] No Phase 2 checkpoint at {PHASE2_CHECKPOINT}")
elif not TEST_SET_FILE:
    print("[!] No test H5 file.")
else:
    !python scripts/evaluate.py \
        --weights {PHASE2_CHECKPOINT} \
        --h5_file {TEST_SET_FILE} \
        --config configs/extensibility_v2.yaml \
        --results_dir {PHASE2_EVAL_DIR}

Using device: cuda
/kaggle/working/hanomi-repo/scripts/evaluate.py:128: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  full_loader = DataLoader(full_dataset, batch_size=args.batch_size, shuffle=False)
Loaded 8632 graphs for evaluation.

--- Face-Level Segmentation Report ---
Segmentation Eval: 100%|███████████████████| 8632/8632 [00:23<00:00, 370.11it/s]

--- Face-Level Segmentation Report ---
              precision    recall  f1-score   support

           0       0.82      0.68      0.74      3487
           1       0.99      0.99      0.99      3723
           2       0.88      0.81      0.84     10965
           3       0.86      0.86      0.86     14723
           4       0.89      0.94      0.92     21871
           5       0.86      0.86      0.86      2450
           6       0.87      0.90      0.89      3935
           7       0.96      0.96      0.96      1327
           8       0.69      0.57      0.62      6797
           9       0.97      

### Step 7: Phase Comparison

| Metric | Phase 1 | Phase 2 | Delta |
|--------|---------|---------|-------|
| F1     | —       | —       | —     |
| Prec   | —       | —       | —     |
| Recall | —       | —       | —     |

In [9]:
import json


def load_metrics(path):
    try:
        with open(path) as f:
            return json.load(f)
    except FileNotFoundError:
        return None


p1 = load_metrics(f"{EVAL_DIR}/metrics.json")
p2 = load_metrics(f"{PHASE2_EVAL_DIR}/metrics.json")

print("=" * 60)
print("PHASE COMPARISON")
print("=" * 60)
if p1:
    print(f"Phase 1 (through_hole + blind_hole): F1={p1['mean_f1']:.4f}  P={p1['mean_precision']:.4f}  R={p1['mean_recall']:.4f}")
else:
    print(f"Phase 1: metrics not found at {EVAL_DIR}/metrics.json")
if p2:
    print(f"Phase 2 (+ rectangular_pocket):      F1={p2['mean_f1']:.4f}  P={p2['mean_precision']:.4f}  R={p2['mean_recall']:.4f}")
else:
    print(f"Phase 2: metrics not found at {PHASE2_EVAL_DIR}/metrics.json")
if p1 and p2:
    print(f"Delta F1: {p2['mean_f1'] - p1['mean_f1']:+.4f}")
print("=" * 60)

PHASE COMPARISON
Phase 1 (through_hole + blind_hole): F1=0.8290  P=0.7709  R=0.9622
Phase 2 (+ rectangular_pocket):      F1=0.8104  P=0.7527  R=0.9450
Delta F1: -0.0186


### Step 8: Ablation Runs

Trains five variants with different config overrides. Output is streamed in real time
via `Popen`; OOM errors are detected in the captured stderr and trigger a batch-size retry.

In [10]:
import subprocess
import sys
import threading
from pathlib import Path

ABLATION_CONFIGS = [
    ("A_full",           "configs/ablations/A_full.yaml"),
    ("B_no_contrastive", "configs/ablations/B_no_contrastive.yaml"),
    ("C_no_edge_features","configs/ablations/C_no_edge_features.yaml"),
    ("D_1hop",           "configs/ablations/D_1hop.yaml"),
    ("E_3hop",           "configs/ablations/E_3hop.yaml"),
]
ABLATION_BATCH_SIZES   = [32, 16, 8]
ABLATION_SKIP_COMPLETE = True


def _run_streaming(cmd):
    """Run cmd, stream stdout live, capture stderr. Returns (returncode, stderr_text)."""
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
    stderr_chunks = []

    def _drain_stderr():
        for line in proc.stderr:
            stderr_chunks.append(line)

    t = threading.Thread(target=_drain_stderr, daemon=True)
    t.start()
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    t.join()
    return proc.returncode, "".join(stderr_chunks)


if not H5_DATASET_PATH:
    print("[!] No dataset path; skipping ablation runs.")
else:
    for tag, cfg in ABLATION_CONFIGS:
        out_dir   = f"results/ablations/{tag}"
        best_ckpt = Path(out_dir) / "checkpoints" / "best.pt"
        if ABLATION_SKIP_COMPLETE and best_ckpt.exists():
            print(f"[✓] {tag} already complete; skipping")
            continue
        print(f"\n{'=' * 52}\n[*] Ablation: {tag}\n{'=' * 52}")
        success = False
        for batch_size in ABLATION_BATCH_SIZES:
            cmd = [
                "python", "scripts/train.py",
                "--h5_dir",     H5_DATASET_PATH,
                "--config",     cfg,
                "--output_dir", out_dir,
                "--batch_size", str(batch_size),
                "--early_stopping",
            ]
            rc, stderr = _run_streaming(cmd)
            if rc == 0:
                success = True
                break
            if "out of memory" in stderr.lower() or "cuda" in stderr.lower():
                print(f"[!] OOM at batch_size={batch_size}; retrying with batch_size={batch_size // 2}")
                continue
            print(f"[!] Non-OOM failure for {tag}; stderr:\n{stderr}")
            break
        if not success:
            print(f"[!] {tag} did not complete")


[*] Ablation: A_full
10:46:02 | INFO | train | Output dir: results/ablations/A_full
10:46:02 | INFO | train | Device: cuda
10:46:02 | INFO | train | Feature types: ['through_hole', 'blind_hole'] → label IDs: [1, 12]
10:57:57 | INFO | train | Dataset sizes — train: 19290  val: 4118  test: 4108
10:57:57 | INFO | train | Model params: 116,634
10:57:57 | INFO | train | Early stopping enabled: patience=10, min_delta=0.0010
10:59:02 | INFO | train | Epoch   1/50 | train_loss=2.7065 | val_loss=1.2228 | val_acc=0.628 | lr=2.00e-04 | 65.2s
10:59:03 | INFO | train |   ✓ Saved best checkpoint (val_loss=1.2228)
11:00:06 | INFO | train | Epoch   2/50 | train_loss=0.9872 | val_loss=0.9695 | val_acc=0.680 | lr=4.00e-04 | 63.5s
11:00:06 | INFO | train |   ✓ Saved best checkpoint (val_loss=0.9695)
11:01:10 | INFO | train | Epoch   3/50 | train_loss=0.7056 | val_loss=0.6299 | val_acc=0.782 | lr=6.00e-04 | 63.7s
11:01:10 | INFO | train |   ✓ Saved best checkpoint (val_loss=0.6299)
11:02:14 | INFO | trai

### Step 9: Baseline Evaluation — Rule-Based and Optional LLM

In [3]:
  import os                              
  from pathlib import Path                                                                                                                                                                                                                       
                                                                                                                                  
  # Re-anchor working directory to the repo
  WORKING_REPO_PATH = Path("/kaggle/working/hanomi-repo")                                                                                                                                                                                        
  if (WORKING_REPO_PATH / "scripts" / "run_baselines.py").exists():
      os.chdir(WORKING_REPO_PATH)                                                                                                                                                                                                                
      print(f"[✓] cwd restored: {os.getcwd()}")                                                                                   
  else:                                                                                                                                                                                                                                          
      print(f"[!] Repo not found at {WORKING_REPO_PATH}")  

[✓] cwd restored: /kaggle/working/hanomi-repo


In [4]:
import h5py
import numpy as np

with h5py.File(TEST_SET_FILE, "r") as f:
  # peek at the first batch group
  batch_key = list(f.keys())[0]
  grp = f[batch_key]
  print("Keys in batch:", list(grp.keys()))
  V1 = np.array(grp["V_1"])
  print("V1 shape:", V1.shape)
  print("V1 col 4 (surface_type) first 10:", V1[:10, 4].tolist())
  labels = np.array(grp["labels"])
  print("labels first 10:", labels[:10].tolist())

Keys in batch: ['A_1_idx', 'A_1_shape', 'A_1_values', 'A_2_idx', 'A_2_shape', 'A_2_values', 'A_3_idx', 'A_3_shape', 'A_3_values', 'A_4_idx', 'A_4_shape', 'A_4_values', 'CAD_model', 'E_1_idx', 'E_1_shape', 'E_1_values', 'E_2_idx', 'E_2_shape', 'E_2_values', 'E_3_idx', 'E_3_shape', 'E_3_values', 'V_1', 'V_2', 'facet_labels', 'idx', 'labels']
V1 shape: (614, 5)
V1 col 4 (surface_type) first 10: [0.09090901166200638, 0.09090901166200638, 0.09090901166200638, 0.09090901166200638, 0.09090901166200638, 0.09090901166200638, 0.09090901166200638, 0.18181802332401276, 0.09090901166200638, 0.09090901166200638]
labels first 10: [24.0, 24.0, 24.0, 9.0, 24.0, 9.0, 24.0, 1.0, 14.0, 14.0]


In [6]:
import h5py
import numpy as np

with h5py.File(TEST_SET_FILE, "r") as f:
  batch_key = list(f.keys())[0]
  grp = f[batch_key]
  V1     = np.array(grp["V_1"])
  labels = np.array(grp["labels"])

  # find a through_hole face
  th_idx = int(np.where(labels == 1)[0][0])
  print(f"through_hole face index: {th_idx}")
  print(f"  surface_type raw: {V1[th_idx, 4]:.4f}  decoded: {round(V1[th_idx, 4] * 11)}")

  # load edge indices
  def load_ei(grp, key):
      idx = np.array(grp[f"{key}_idx"])
      return idx if idx.ndim == 2 else idx.reshape(2, -1)

  a1  = load_ei(grp, "A_1")   # all edges
  e1  = load_ei(grp, "E_1")   # convex
  e2  = load_ei(grp, "E_2")   # concave

  # neighbors of through_hole face in A_1
  nbr_cols = np.where(a1[0] == th_idx)[0]
  print(f"\nNeighbors of face {th_idx} (via A_1):")
  for c in nbr_cols:
      dst = a1[1, c]
      stype = round(V1[dst, 4] * 11)
      lbl   = labels[dst]
      in_e1 = ((e1[0] == th_idx) & (e1[1] == dst)).any()
      in_e2 = ((e2[0] == th_idx) & (e2[1] == dst)).any()
      conv  = "CONVEX" if in_e1 else ("CONCAVE" if in_e2 else "SMOOTH")
      print(f"  → face {dst}  surf_type={stype}  label={lbl:.0f}  edge={conv}")

through_hole face index: 7
  surface_type raw: 0.1818  decoded: 2

Neighbors of face 7 (via A_1):


In [16]:
if not TEST_SET_FILE:                                                                                                                                                                                                                          
  print("[!] No test file found; skipping baselines.")                                                                                                                                                                                       
else:                                                                                                                                                                                                                                          
  import sys, os, time                                                                                                                                                                                                                       
  sys.path.insert(0, os.getcwd())                                                                                                                                                                                                            
  from collections import deque                                                                                               
  from tqdm import tqdm                                                                                                                                                                                                                      
  import src.baselines.rule_based as rb                                                                                       
  from src.data.h5_dataset import MFCADPlusPlusDataset                                                                                                                                                                                       
  from src.evaluation.metrics import instance_f1                                                                                                                                                                                             
  from src.evaluation.results_logger import ResultsLogger, ModelResult                                                                                                                                                                       
                                                                                                                                                                                                                                             
  rb._get_surface_type = lambda data, fi: int(data.x[fi, 4].item() * 11)                                                                                                                                                                     
                                                                                                                                                                                                                                             
  def _get_all_neighbors(data, face_idx):                                                                                                                                                                                                    
      nbrs = []                                                                                                               
      for col in range(data.edge_index.size(1)):
          if int(data.edge_index[0, col]) != face_idx:                                                                                                                                                                                       
              continue
          dst = int(data.edge_index[1, col])                                                                                                                                                                                                 
          nbrs.append(dst)                                                                                                                                                                                                                   
      return nbrs
                                                                                                                                                                                                                                             
  def match_through_hole_v2(data):                                                                                                                                                                                                           
      instances = []
      visited = set()                                                                                                                                                                                                                        
      for fi in range(data.num_nodes):                                                                                        
          if fi in visited:                                                                                                                                                                                                                  
              continue
          if rb._get_surface_type(data, fi) != 1:  # CYLINDER                                                                                                                                                                                
              continue                                                                                                        
          nbrs = _get_all_neighbors(data, fi)
          planar_caps = [n for n in nbrs if rb._get_surface_type(data, n) == 0]  # PLANE                                                                                                                                                     
          if len(planar_caps) >= 2:                                                                                                                                                                                                          
              visited.add(fi)                                                                                                                                                                                                                
              instances.append({"face_ids": [fi], "confidence": 0.85})                                                                                                                                                                       
      return instances                                                                                                                                                                                                                       

  def _build_gt(data, label_id):                                                                                                                                                                                                             
      target = set((data.y == label_id).nonzero(as_tuple=True)[0].tolist())                                                   
      if not target: return []                                                                                                                                                                                                               
      adj = {i: [] for i in range(data.num_nodes)}                                                                            
      for c in range(data.edge_index.size(1)):                                                                                                                                                                                               
          adj[int(data.edge_index[0,c])].append(int(data.edge_index[1,c]))                                                    
      visited, instances = set(), []                                                                                                                                                                                                         
      for s in target:                                                                                                                                                                                                                       
          if s in visited: continue                                                                                                                                                                                                          
          comp, q = [], deque([s])                                                                                                                                                                                                           
          visited.add(s)                                                                                                      
          while q:                                                                                                                                                                                                                           
              n = q.popleft(); comp.append(n)
              for nb in adj[n]:                                                                                                                                                                                                              
                  if nb not in visited and nb in target:                                                                      
                      visited.add(nb); q.append(nb)                                                                                                                                                                                          
          instances.append({"face_ids": sorted(comp)})
      return instances                                                                                                                                                                                                                       
                                                                                                                              
  dataset  = MFCADPlusPlusDataset(TEST_SET_FILE)                                                                                                                                                                                             
  label_id = 1
  logger   = ResultsLogger("results/baselines")                                                                                                                                                                                              
  count    = 0                                                                                                                                                                                                                               
  debugged = False
                                                                                                                                                                                                                                             
  for i, data in enumerate(tqdm(dataset, desc="rule_based_v2")):                                                              
      if count >= 200: break                                                                                                                                                                                                                 
      gt = _build_gt(data, label_id)                                                                                                                                                                                                         
      if not gt: continue
                                                                                                                                                                                                                                             
      t0   = time.time()                                                                                                      
      pred = match_through_hole_v2(data)
      ms   = (time.time() - t0) * 1000                                                                                                                                                                                                       

      if not debugged:                                                                                                                                                                                                                       
          th_faces = [f for inst in gt for f in inst["face_ids"]]                                                             
          print(f"\n[DEBUG] model {i}: GT={th_faces[:5]}, pred={[p['face_ids'] for p in pred[:3]]}")                                                                                                                                         
          for fi in th_faces[:2]:                                                                                                                                                                                                            
              nbrs = _get_all_neighbors(data, fi)                                                                                                                                                                                            
              planes = [n for n in nbrs if rb._get_surface_type(data, n) == 0]                                                                                                                                                               
              print(f"  face {fi}: all_nbrs={nbrs}, plane_nbrs={planes}")                                                                                                                                                                    
          debugged = True                                                                                                     
                                                                                                                                                                                                                                             
      p, r, f1 = instance_f1(pred, gt)                                                                                        
      logger.log(ModelResult(                                                                                                                                                                                                                
          run_id="baselines", model_file=str(i), feature_type="through_hole",                                                                                                                                                                
          method="rule_based", predicted_instances=pred, gt_instances=gt,                                                                                                                                                                    
          precision=p, recall=r, f1=f1, inference_ms=ms                                                                                                                                                                                      
      ))                                                                                                                                                                                                                                     
      count += 1                                                                                                                                                                                                                             
                                                                                                                              
  logger.print_summary_table()                                                                                                                                                                                                               
  logger.save()


rule_based_v2:   1%|          | 54/8632 [00:00<00:16, 534.00it/s]


[DEBUG] model 0: GT=[20, 7], pred=[[7], [20], [26]]
  face 20: all_nbrs=[1, 4, 20], plane_nbrs=[1, 4]
  face 7: all_nbrs=[0, 7, 18], plane_nbrs=[0, 18]


rule_based_v2:   8%|▊         | 687/8632 [00:00<00:10, 736.57it/s]


model                          method           F1      P      R       ms
─────────────────────────────────────────────────────────────────────────
26                             rule_based    1.000  1.000  1.000      1.5
38                             rule_based    1.000  1.000  1.000      1.0
39                             rule_based    1.000  1.000  1.000      0.8
54                             rule_based    1.000  1.000  1.000      0.4
57                             rule_based    1.000  1.000  1.000      1.1
105                            rule_based    1.000  1.000  1.000      0.9
122                            rule_based    1.000  1.000  1.000      0.6
142                            rule_based    1.000  1.000  1.000      1.2
179                            rule_based    1.000  1.000  1.000      0.9
182                            rule_based    1.000  1.000  1.000      0.7
192                            rule_based    1.000  1.000  1.000      1.5
224                            rule_b

### Step 10: Inference Benchmark and Cost Estimate

In [4]:
import subprocess, threading                                                                                                                                                                                                                   
                                                                                                                                                                                                                                             
def _run_streaming(cmd):                                                                                                                                                                                                                       
  proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)                                                                                                                                         
  stderr_chunks = []                                                                                                                                                                                                                         
  def _drain():    
      for line in proc.stderr:                                                                                                                                                                                                               
          stderr_chunks.append(line)                                                                                                                                                                                                         
  t = threading.Thread(target=_drain, daemon=True)
  t.start()                                                                                                                                                                                                                                  
  for line in proc.stdout:                                                                                                    
      print(line, end="", flush=True)                                                                                                                                                                                                        
  proc.wait()
  t.join()                                                                                                                                                                                                                                   
  return proc.returncode, "".join(stderr_chunks)  

In [5]:
import subprocess
import time
import yaml
import torch

if not os.path.exists(CHECKPOINT):
    print(f"[!] Missing checkpoint at {CHECKPOINT} — run training first.")
else:
    print("[*] Running benchmark...")
    cmd = [
        "python", "scripts/benchmark_inference.py",
        "--config",     "configs/counterbored_hole.yaml",
        "--checkpoint", CHECKPOINT,
        "--h5_dir",     str(H5_DATASET_PATH or ""),
        "--n_samples",  "200",
        "--estimate_cost", "10000",
    ]
    rc, stderr = _run_streaming(cmd)

    if rc != 0:
        print("[!] benchmark_inference.py failed; running notebook fallback...")
        if stderr:
            print(stderr)

        from src.models.feature_recognizer import FeatureRecognizer
        from src.data.dataloader import make_dataloaders

        with open("configs/counterbored_hole.yaml") as f:
            cfg = yaml.safe_load(f)

        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        ckpt   = torch.load(CHECKPOINT, map_location=device)
        model  = FeatureRecognizer(cfg).to(device)
        model.load_state_dict(ckpt["model"])
        model.eval()

        loaders = make_dataloaders(
            h5_dir=H5_DATASET_PATH,
            feature_types=cfg["feature_types"],
            batch_size=1,
            num_workers=cfg["data"].get("num_workers", 4),
            seed=42,
        )
        times = []
        with torch.no_grad():
            for i, data in enumerate(loaders["test"]):
                if i >= 200:
                    break
                data = data.to(device)
                t0 = time.time()
                _ = model(data)
                times.append((time.time() - t0) * 1000.0)

        if times:
            mean_ms    = sum(times) / len(times)
            throughput = 1000.0 / mean_ms if mean_ms > 0 else 0.0
            print(f"Fallback: mean={mean_ms:.2f} ms/model  throughput={throughput:.1f} models/s  gpu_s/query={mean_ms/1000:.4f}")
        else:
            print("[!] No samples benchmarked.")

NameError: name 'os' is not defined

In [22]:
import subprocess, threading, time, yaml, torch, os                                                                                                                                                                                            
                                                                                                                              
def _run_streaming(cmd):                                                                                                                                                                                                                       
  proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True, bufsize=1)
  stderr_chunks = []                                                                                                                                                                                                                         
  def _drain():                                                                                                               
      for line in proc.stderr:                                                                                                                                                                                                               
          stderr_chunks.append(line)                                                                                          
  t = threading.Thread(target=_drain, daemon=True)
  t.start()                                                                                                                                                                                                                                  
  for line in proc.stdout:
      print(line, end="", flush=True)                                                                                                                                                                                                        
  proc.wait()                                                                                                                 
  t.join()
  return proc.returncode, "".join(stderr_chunks)                                                                                                                                                                                             

if not os.path.exists(CHECKPOINT):                                                                                                                                                                                                             
  print(f"[!] Missing checkpoint at {CHECKPOINT} — run training first.")                                                      
else:                                                                                                                                                                                                                                          
  print("[*] Running benchmark (inline)...")                                                                                                                                                                                                 
  import sys                                                                                                                                                                                                                                 
  sys.path.insert(0, os.getcwd())                                                                                                                                                                                                            
                                                                                                                              
  from src.models.feature_recognizer import FeatureRecognizer                                                                                                                                                                                
  from src.data.dataloader import make_dataloaders
                                                                                                                                                                                                                                             
  with open("configs/counterbored_hole.yaml") as f:                                                                                                                                                                                          
      cfg = yaml.safe_load(f)
                                                                                                                                                                                                                                             
  device = torch.device("cuda" if torch.cuda.is_available() else "cpu")                                                       
  ckpt   = torch.load(CHECKPOINT, map_location=device)
  model  = FeatureRecognizer(cfg).to(device)                                                                                                                                                                                                 
  model.load_state_dict(ckpt["model"])
  model.eval()                                                                                                                                                                                                                               
                                                                                                                              
  loaders = make_dataloaders(                                                                                                                                                                                                                
      h5_dir=H5_DATASET_PATH,                                                                                                 
      feature_types=cfg["feature_types"],                                                                                                                                                                                                    
      batch_size=1,
      num_workers=0,                                                                                                                                                                                                                         
      seed=42,                                                                                                                                                                                                                               
  )
                                                                                                                                                                                                                                             
  times = []                                                                                                                  
  with torch.no_grad():
      for i, data in enumerate(loaders["test"]):
          if i >= 200:                                                                                                                                                                                                                       
              break
          data = data.to(device)                                                                                                                                                                                                             
          t0 = time.time()                                                                                                    
          _ = model(data)
          if device.type == "cuda":
              torch.cuda.synchronize()                                                                                                                                                                                                       
          times.append((time.time() - t0) * 1000.0)
                                                                                                                                                                                                                                             
  if times:                                                                                                                                                                                                                                  
      mean_ms    = sum(times) / len(times)
      throughput = 1000.0 / mean_ms                                                                                                                                                                                                          
      gpu_s      = mean_ms / 1000.0                                                                                           
      daily_10k  = gpu_s * 10_000 / 3600                                                                                                                                                                                                     
      print(f"\nBenchmark over {len(times)} models:")                                                                                                                                                                                        
      print(f"  Mean latency : {mean_ms:.2f} ms/model")                                                                                                                                                                                      
      print(f"  Throughput   : {throughput:.1f} models/s")                                                                                                                                                                                   
      print(f"  GPU-s/query  : {gpu_s:.4f}")                                                                                                                                                                                                 
      print(f"  Est. GPU-hrs for 10k queries/day: {daily_10k:.2f} h")                                                                                                                                                                        
                                                                                                                                                                                                                                             
      import json                                                                                                             
      from pathlib import Path                                                                                                                                                                                                               
      cost_path = Path(RUN_DIR) / "cost_estimate.json"                                                                        
      cost_path.parent.mkdir(parents=True, exist_ok=True)                                                                                                                                                                                    
      with open(cost_path, "w") as f:
          json.dump({                                                                                                                                                                                                                        
              "mean_ms": mean_ms,                                                                                             
              "throughput_per_sec": throughput,                                                                                                                                                                                              
              "gpu_seconds_per_query": gpu_s,                                                                                                                                                                                                
              "estimated_gpu_hours_10k_queries_per_day": daily_10k,
              "n_samples": len(times),                                                                                                                                                                                                       
          }, f, indent=2)                                                                                                     
      print(f"  Saved to {cost_path}")                                                                                                                                                                                                       
  else:                                                                                                                       
      print("[!] No samples benchmarked.")                  

[*] Running benchmark (inline)...

Benchmark over 200 models:
  Mean latency : 3.50 ms/model
  Throughput   : 286.1 models/s
  GPU-s/query  : 0.0035
  Est. GPU-hrs for 10k queries/day: 0.01 h
  Saved to results/runs/run_001/cost_estimate.json


### Step 11: Data Quality Validation

In [23]:
if H5_DATASET_PATH:
    !python scripts/validate_data.py \
        --h5_dir {H5_DATASET_PATH} \
        --output_dir results/data_quality
else:
    print("[!] Missing H5 path; skipping data validation.")


No quality issues found! 🎉


In [ ]:
if not H5_DATASET_PATH:
  print("[!] Missing H5 path; skipping data validation.")
else:
  import sys, os
  sys.path.insert(0, os.getcwd())
  from pathlib import Path
  from src.data.h5_dataset import MFCADPlusPlusDataset

  results_dir = Path("results/data_quality")
  results_dir.mkdir(parents=True, exist_ok=True)

  for split, fname in [
      ("training", next((f for f in ["training_MFCAD++.h5", "training_MFCAD.h5"] if (Path(H5_DATASET_PATH) / f).exists()), None)),
      ("val",      next((f for f in ["val_MFCAD++.h5",      "val_MFCAD.h5"]      if (Path(H5_DATASET_PATH) / f).exists()), None)),
      ("test",     next((f for f in ["test_MFCAD++.h5",     "test_MFCAD.h5"]     if (Path(H5_DATASET_PATH) / f).exists()), None)),
  ]:
      if fname is None:
          print(f"[!] {split}: file not found, skipping")
          continue
      h5_path = Path(H5_DATASET_PATH) / fname
      ds = MFCADPlusPlusDataset(str(h5_path))
      n_empty = sum(1 for d in ds if d.num_nodes == 0)
      n_no_edges = sum(1 for d in ds if d.edge_index.size(1) == 0)
      n_bad_labels = sum(1 for d in ds if (d.y < 0).any() or (d.y > 24).any())
      print(f"[{split}] {len(ds)} models — empty:{n_empty}  no_edges:{n_no_edges}  bad_labels:{n_bad_labels}")

  print(f"\n[✓] Validation complete. Results dir: {results_dir}")

In [ ]:
if not H5_DATASET_PATH:                                                                                                                                                                                                                        
  print("[!] Missing H5 path; skipping data validation.")                                                                                                                                                                                    
else:                                                                                                                                                                                                                                          
  import sys, os                                                                                                              
  sys.path.insert(0, os.getcwd())
  from pathlib import Path                                                                                                                                                                                                                   
  from src.data.h5_dataset import MFCADPlusPlusDataset
                                                                                                                                                                                                                                             
  for split, candidates in [                                                                                                  
      ("training", ["training_MFCAD++.h5", "training_MFCAD.h5"]),
      ("val",      ["val_MFCAD++.h5",      "val_MFCAD.h5"]),                                                                                                                                                                                 
      ("test",     ["test_MFCAD++.h5",     "test_MFCAD.h5"]),
  ]:                                                                                                                                                                                                                                         
      fname = next((f for f in candidates if (Path(H5_DATASET_PATH) / f).exists()), None)                                     
      if fname is None:                                                                                                                                                                                                                      
          print(f"[!] {split}: not found"); continue                                                                          
      ds = MFCADPlusPlusDataset(str(Path(H5_DATASET_PATH) / fname))                                                                                                                                                                          
      limit = min(200, len(ds))                                                                                                                                                                                                              
      n_empty = n_no_edges = n_bad_labels = 0
      for i in range(limit):                                                                                                                                                                                                                 
          d = ds[i]                                                                                                           
          if d.num_nodes == 0: n_empty += 1                                                                                                                                                                                                  
          if d.edge_index.size(1) == 0: n_no_edges += 1                                                                       
          if (d.y < 0).any() or (d.y > 24).any(): n_bad_labels += 1                                                                                                                                                                          
      print(f"[{split}] {len(ds)} total, checked {limit} — empty:{n_empty}  no_edges:{n_no_edges}  bad_labels:{n_bad_labels}")

### Step 12: Package Outputs for Download

Bundles all heavy-compute artifacts into a single archive.

In [28]:
archive = "results/packages/hanomi_results.tar.gz"
!mkdir -p results/packages
!tar -czf {archive} \
    {EVAL_DIR} \
    {PHASE2_EVAL_DIR} \
    results/ablations \
    results/baselines \
    results/baselines_eval \
    results/data_quality \
    {RUN_DIR}/checkpoints \
    {PHASE2_DIR}/checkpoints \
    2>/dev/null || true
!ls -lh {archive} 2>/dev/null || echo '[!] Archive is empty — nothing was produced yet.'

-rw-r--r-- 1 root root 18M Apr 27 22:21 results/packages/hanomi_results.tar.gz
